# 🏠 Mumbai Real Estate Data Science & Valuation Model

This Jupyter Notebook walks through the complete end-to-end data science workflow for analyzing Mumbai housing prices and training a machine learning model to estimate property valuations.

## Table of Contents
1. **Exploratory Data Analysis (EDA)**: Inspecting distributions, correlations, and outliers.
2. **Feature Engineering & Preprocessing**: Scaling numerical inputs and encoding locality tags.
3. **Model Selection**: Comparing Linear Regression, Decision Tree, and Random Forest Regressor models.
4. **Model Evaluation**: Metrics breakdown ($R^2$ Score, MAE, RMSE).
5. **Feature Importance Analysis**: Finding the main value drivers in the Mumbai real estate market.
6. **Model Serialization**: Saving the trained pipeline for the Streamlit dashboard.

### Step 1: Import Libraries & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

In [ ]:
# Load the dataset
data_file = 'mumbai_house_data.csv'
if not os.path.exists(data_file):
    print("mumbai_house_data.csv not found! Please run python data_generator.py first to create it.")
else:
    df = pd.read_csv(data_file)
    print(f"Dataset successfully loaded. Shape: {df.shape}")

### Step 2: Exploratory Data Analysis

In [ ]:
# Display metadata and stats
df.info()

In [ ]:
# Summary statistics
df.describe()

In [ ]:
# Display first few rows
df.head()

In [ ]:
# 1. Distribution of Property Prices
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.histplot(df['Price_Lakhs'], kde=True, bins=30, color='royalblue', ax=ax[0])
ax[0].set_title('Distribution of House Prices in Lakhs (INR)')
ax[0].set_xlabel('Price (Lakhs INR)')

# 2. Distribution of Property Sizes
sns.histplot(df['Area_SqFt'], kde=True, bins=30, color='seagreen', ax=ax[1])
ax[1].set_title('Distribution of Property Size (SqFt)')
ax[1].set_xlabel('Area (SqFt)')

plt.tight_layout()
plt.show()

In [ ]:
# 3. Boxplot: Pricing across local areas
plt.figure(figsize=(14, 7))
sns.boxplot(data=df, x='Price_Lakhs', y='Locality', palette='coolwarm')
plt.title('Property Valuation Ranges by Locality in Mumbai')
plt.xlabel('Price (Lakhs INR)')
plt.ylabel('Locality')
plt.show()

In [ ]:
# 4. Price vs Area Colored by Locality
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df, x='Area_SqFt', y='Price_Lakhs', hue='Locality', alpha=0.7, palette='tab10')
plt.title('Area (SqFt) vs Price (Lakhs) by Locality')
plt.xlabel('Area (SqFt)')
plt.ylabel('Price (Lakhs INR)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# 5. Heatmap for Correlation Matrix of Numeric Values
plt.figure(figsize=(10, 8))
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

### Step 3: Feature Preprocessing and Splits

In [ ]:
# Define independent features X and dependent target y
X = df.drop(columns=['Price_Lakhs'])
y = df['Price_Lakhs']

# Split datasets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

In [ ]:
# Prepare Column Preprocessor
categorical_cols = ['Locality']
numerical_cols = ['Area_SqFt', 'BHK', 'Bathrooms', 'Property_Age_Years', 'Floor_Num', 'Total_Floors']
binary_cols = ['Parking_Available', 'Swimming_Pool', 'Gym_Available', 'Lift_Available']

preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', StandardScaler(), numerical_cols),
        ('bin', 'passthrough', binary_cols)
    ]
)

### Step 4: Model Training and Selection

We will test three regression algorithms:
1. **Linear Regression**: Baseline linear model.
2. **Decision Tree Regressor**: Tree-based model (non-linear).
3. **Random Forest Regressor**: Ensemble of Decision Trees (highly robust).

In [ ]:
# Dictionary to hold pipelines
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(max_depth=10, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=15, min_samples_split=5, random_state=42)
}

results = {}

for name, model in models.items():
    # Create Pipeline
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Train pipeline
    pipeline.fit(X_train, y_train)
    
    # Predict
    y_pred = pipeline.predict(X_test)
    
    # Evaluate
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    results[name] = {
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Pipeline': pipeline
    }
    
    print(f"=== {name} ===")
    print(f"Mean Absolute Error (MAE): {mae:.2f} Lakhs")
    print(f"Root Mean Squared Error (RMSE): {rmse:.2f} Lakhs")
    print(f"R-squared Score (R2): {r2:.4f} ({r2 * 100:.2f}%)\n")

In [ ]:
# Plot Model Performance Comparison
model_names = list(results.keys())
r2_scores = [results[m]['R2'] * 100 for m in model_names]
rmse_scores = [results[m]['RMSE'] for m in model_names]

fig, ax1 = plt.subplots(figsize=(10, 5))

# Bar chart for R2
color = 'royalblue'
ax1.set_xlabel('Regression Model')
ax1.set_ylabel('R2 Score (%)', color=color)
bars = ax1.bar(model_names, r2_scores, color=color, alpha=0.6, width=0.4, align='center')
ax1.tick_params(axis='y', labelcolor=color)
ax1.set_ylim(0, 100)

# Annotate R2 values
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2.0, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

# Line chart for RMSE
ax2 = ax1.twinx()
color = 'crimson'
ax2.set_ylabel('RMSE (Lakhs)', color=color)
ax2.plot(model_names, rmse_scores, color=color, marker='o', linewidth=2.5, markersize=8)
ax2.tick_params(axis='y', labelcolor=color)

plt.title('Model Evaluation: R2 Score vs. Root Mean Squared Error (RMSE)')
plt.tight_layout()
plt.show()

### Step 5: Feature Importance Analysis (Random Forest)

In [ ]:
# Extract features importance from Random Forest Pipeline
rf_pipeline = results['Random Forest']['Pipeline']
rf_regressor = rf_pipeline.named_steps['regressor']

# Extract categoric feature names from OneHotEncoder
ohe_encoder = rf_pipeline.named_steps['preprocessor'].named_transformers_['cat']
ohe_features = list(ohe_encoder.get_feature_names_out(categorical_cols))

# Complete list of features
all_features = ohe_features + numerical_cols + binary_cols
importances = rf_regressor.feature_importances_

feature_imp_df = pd.DataFrame({
    'Feature': [f.replace('Locality_', 'Locality: ') for f in all_features],
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# Plot top 10 feature importances
plt.figure(figsize=(12, 6))
sns.barplot(data=feature_imp_df.head(10), x='Importance', y='Feature', palette='viridis')
plt.title('Top 10 Most Predictive Factors in Mumbai Property Pricing')
plt.xlabel('Feature Weight / Importance')
plt.ylabel('Feature')
plt.show()

### Step 6: Save the Trained Model Pipeline

In [ ]:
# Save Random Forest pipeline to best_model.pkl
best_pipeline = results['Random Forest']['Pipeline']
with open('best_model.pkl', 'wb') as f:
    pickle.dump(best_pipeline, f)

print("Model pipeline saved successfully to 'best_model.pkl' for deployment.")